In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

### Langchain Concepts

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [15]:
template = "You are an expert stock market trader. Explain {topic} to user in simple terms with use of emojis."
prompt = ChatPromptTemplate.from_template(template=template)

In [16]:
parser = StrOutputParser()
chain = prompt | chat_model | parser

In [17]:
from rich import print

In [18]:
results = chain.invoke({"topic": "Derivatives"})

In [20]:
# print(results)

## Static Workflow

In [31]:
from langgraph.graph import StateGraph, START, END

#### State of Graph

In [46]:
from typing import TypedDict


class AreaState(TypedDict):
    lenght_rect: float = None
    breadth_rect: float = None
    height_rect: float = None
    area_rect: float = None
    bhk_size: str = None

In [47]:
graph = StateGraph(state_schema=AreaState)

#### Nodes of Graph

In [48]:
def CalcArea(state: AreaState) -> AreaState:
    l = state["lenght_rect"]
    b = state["breadth_rect"]
    h = state["height_rect"]
    area = l * b * h
    state["area_rect"] = area
    return state

In [49]:
def ClassifyBHK(state: AreaState) -> AreaState:
    area_rect = state["area_rect"]
    if 0 <= area_rect < 10000:
        state["bhk_size"] = "2_BHK"
    elif 10000 <= area_rect < 20000:
        state["bhk_size"] = "3_BHK"
    else:
        state["bhk_size"] = "4_BHK"
    return state

In [50]:
graph.add_node("AreaCalculate", CalcArea)
graph.add_node("ClassifyBHK", ClassifyBHK)

#### Edges of the Graph

In [51]:
graph.add_edge(START, "AreaCalculate")
graph.add_edge("AreaCalculate", "ClassifyBHK")
graph.add_edge("ClassifyBHK", END)

In [52]:
workflow = graph.compile()

In [54]:
final_state = workflow.invoke(
    {"lenght_rect": 12.3, "breadth_rect": 32.3, "height_rect": 43.5}
)

In [55]:
print(
    f"Area of the Rectangle is {final_state['area_rect']} Having Size: {final_state['bhk_size']}"
)

Area of the Rectangle is 17282.114999999998 Having Size: 3_BHK

### LLM Based Workflow

In [91]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",
)

model = ChatHuggingFace(llm=llm)

In [84]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate,
    AIMessagePromptTemplate,
)

In [85]:
system = SystemMessagePromptTemplate.from_template("""
You are an expert Data Scientist with over 10 years of experience.

Your goal is to help the user develop:
- Practical understanding
- Mathematical understanding
- Intuition behind the concepts

Always:
- Explain from first principles.
- Use simple language.
- Use real-world examples.
- Explain mathematical notation before using formulas.
- Build intuition before diving into technical details.
""")

In [86]:
human = HumanMessagePromptTemplate.from_template(
    "Give Mathematical Definition and Data Science use of {topic}."
)
ai_msg = AIMessagePromptTemplate.from_template('Hi {name}. Let"s Start Deep Diving.')

In [87]:
prompt = ChatPromptTemplate.from_messages([system, ai_msg, human])

In [98]:
chain = prompt | model | parser

In [99]:
class LlmState(TypedDict):
    name: str = None
    topic: str = None
    answer: str = None

In [100]:
llm_graph = StateGraph(state_schema=LlmState)

In [102]:
def GetAnswer(state: LlmState):
    name = state["name"]
    topic = state["topic"]
    result = chain.invoke({"name": name, "topic": topic})
    state["answer"] = result
    return state

In [103]:
llm_graph.add_node("GetLlmAnswer", GetAnswer)

In [104]:
llm_graph.add_edge(START, "GetLlmAnswer")
llm_graph.add_edge("GetLlmAnswer", END)

In [105]:
llm_workflow = llm_graph.compile()

In [110]:
intial_state = {"name": "gourav", "topic": "Human Reproductive System"}

In [111]:
final_llm_state = llm_workflow.invoke(intial_state)

In [113]:
print(final_llm_state["answer"])

What an interesting topic. I'll do my best to provide a clear and concise explanation.

**Mathematical Definition of the Human Reproductive System:**

From a mathematical perspective, the human reproductive system can be modeled as a complex network of 
interconnected components. Let's break it down:

1. **Input (Inflow)**: Hormones, such as FSH (Follicle-Stimulating Hormone) and LH (Luteinizing Hormone), flow into
the system from the pituitary gland.
2. **Network (System)**: The reproductive system consists of the ovaries, fallopian tubes, uterus, and vas deferens
(in males). We can model each component as a node in a network, with edges representing the flow of hormones and 
fluids between nodes.
3. **Output (Outflow)**: The system produces two primary outputs:
        * In females: Ova (eggs) are released from the ovaries and travel through the fallopian tubes, where 
fertilization can occur.
        * In males: Sperm are produced in the testes and stored in the vas deferens.

Mathematically, we can represent the reproductive system as a nonlinear dynamical system, with the following 
components:

* **State variables**: Concentrations of hormones and fluids within the system
* **Control variables**: External factors influencing the system's behavior (e.g., light, temperature, stress)
* **Output variables**: Ova or sperm production rates

We can use differential equations to model the dynamics of the reproductive system, considering factors like 
feedback loops, oscillations, and nonlinearity.

**Data Science Use of the Human Reproductive System:**

In Data Science, we can apply various techniques to analyze and model the human reproductive system. A few 
potential applications include:

1. **Predictive Modeling**: Using historical data, we can build predictive models to forecast fertility rates, 
menstrual cycles, or sperm count.
2. **Hormone Regulation**: Analyzing hormone levels and their relationships can help understand the underlying 
mechanisms of fertility and develop targeted therapies.
3. **Network Analysis**: Studying the interactions between different components of the reproductive system can 
reveal insights into the system's behavior and identify potential intervention points.
4. **Personalized Medicine**: By considering individual characteristics, such as genetic predispositions and 
medical history, Data Science can help tailor fertility treatments and improve outcomes.

Some potential Data Science tools and techniques for analyzing the human reproductive system include:

* Time series analysis (e.g., ARIMA

## Prompt Chaining Workflow

In [114]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",
)

model_1 = ChatHuggingFace(llm=llm)
model_2 = ChatHuggingFace(llm=llm)

In [115]:
sys_template_1 = """
You are an expert Data Scientist, Machine Learning Engineer, and Technical Writer.

Your responsibility is to design a high-quality technical blog outline for the given topic.

Requirements:
- Target audience: Beginners to Intermediate learners.
- Explain concepts in a logical learning order.
- Cover both intuition and technical depth.
- Include mathematical intuition where relevant.
- Include practical examples and real-world applications.
- Suggest sections where code demonstrations should be added.
- Include common mistakes and best practices.
- End with a concise summary and further reading suggestions.

Return only the blog outline in Markdown format.

Topic:
{topic}
"""

In [147]:
sys_template_2 = """
You are an experienced Data Scientist and Technical Content Writer with over 10 years of experience.

Your task is to write a comprehensive, well-structured, and engaging technical blog based on the provided topic and outline.

Instructions:
- Follow the outline exactly and expand each section thoroughly.
- Write in simple, beginner-friendly language while maintaining technical accuracy.
- Explain concepts from first principles.
- Use real-world examples and analogies wherever appropriate.
- Include mathematical intuition and formulas when relevant, explaining all notation before using it.
- Add practical applications and industry use cases.
- Include Python code snippets where appropriate and explain what each code example demonstrates.
- Use Markdown formatting.
- Use clear headings (H1, H2, H3), bullet points, tables, blockquotes, and code blocks to make the blog visually appealing.
- Highlight important notes, best practices, common mistakes, and key takeaways.
- End the blog with:
  - A concise summary
  - Frequently Asked Questions (FAQs)
  - Suggested next topics for further learning

Topic:
{topic}

Outline:
{outline}
"""

In [148]:
system_prompt_template_1 = SystemMessagePromptTemplate.from_template(
    template=sys_template_1
)
system_prompt_template_2 = SystemMessagePromptTemplate.from_template(
    template=sys_template_2
)

In [149]:
human_prompt_1 = HumanMessagePromptTemplate.from_template(
    "Write Outline for Topic: {topic}"
)

In [150]:
system_1_prompt = ChatPromptTemplate.from_messages(
    [system_prompt_template_1, human_prompt_1]
)

In [151]:
system_2_prompt = ChatPromptTemplate.from_messages([system_prompt_template_2])

In [152]:
outline_chain = system_1_prompt | model_1 | parser
blog_chain = system_2_prompt | model_2 | parser

In [153]:
class BlogAgentState(TypedDict):
    topic: str
    outline: str
    blog: str

In [154]:
blog_graph = StateGraph(BlogAgentState)

In [155]:
def GenerateOutline(state: BlogAgentState) -> BlogAgentState:
    topic = state["topic"]
    outline = outline_chain.invoke({"topic": topic})
    state["outline"] = outline
    return state

In [156]:
def GenerateBlog(state: BlogAgentState) -> BlogAgentState:
    outline = state["outline"]
    topic = state["topic"]
    blog = blog_chain.invoke({"topic": topic, "outline": outline})
    state["blog"] = blog
    return state

In [157]:
blog_graph.add_node("outline", GenerateOutline)
blog_graph.add_node("blog", GenerateBlog)

In [158]:
blog_graph.add_edge(START, "outline")
blog_graph.add_edge("outline", "blog")
blog_graph.add_edge("blog", END)

In [159]:
blog_workflow = blog_graph.compile()

In [162]:
initial_state_blog = {"topic": "gradient descent"}

In [163]:
final_state_blog = blog_workflow.invoke(initial_state_blog)

In [166]:
print(final_state_blog["outline"])

**Gradient Descent Tutorial**
=====================================

### Introduction
---------------

*   **What is Gradient Descent?**
    *   Definition
    *   Intuition: optimzation problem, iterative process
    *   Real-world examples: loss function minimization, cost reduction
*   **Why Gradient Descent?**
    *   Advantages: flexibility, efficiency, robustness
    *   Limitations: convergence issues, local optima

### Mathematical Background
---------------------------

*   **Gradient**
    *   Definition: derivative of a function
    *   Intuition: slope of the function at a point
    *   Mathematical notation: ∇f(x)
*   **Gradient Vector**
    *   Definition: vector of partial derivatives
    *   Intuition: direction of maximum change
    *   Mathematical notation: ∇f(x) = (∂f/∂x1(x), ..., ∂f/∂xn(x))

### Gradient Descent Algorithm
-----------------------------

*   **Basic Gradient Descent**
    *   Definition: iterative process to minimize a loss function
    *   Algorithm: update parameters using the gradient
    *   Mathematical notation: θ = θ - α * ∇J(θ)
*   **Stochastic Gradient Descent**
    *   Definition: minimize loss function using mini-batches
    *   Algorithm: update parameters using the gradient of a mini-batch
    *   Mathematical notation: θ = θ - α * ∇J(θ) using a mini-batch

### Practical Implementation
---------------------------

*   **Choosing the Learning Rate**
    *   Definition: step size of each update
    *   Intuition: balance exploration and exploitation
    *   Mathematical notation: α
*   **Regularization Techniques**
    *   Definition: prevent overfitting
    *   Intuition: add penalty to the loss function
    *   Mathematical notation: L1, L2 regularization
*   **Code Demonstration** [Python code]
    *   Implement basic gradient descent
    *   Use a real-world dataset (e.g., Boston Housing Dataset)

### Advanced Topics
------------------

*   **Convergence Issues**
    *   Definition: problems with gradient descent convergence
    *   Intuition: slow convergence or convergence to a local optima
    *   Mathematical notation: convergence rates, stopping criteria
*   **Momentum and Nesterov Acceleration**
    *   Definition: techniques to improve

In [167]:
print(final_state_blog["blog"])

**Gradient Descent Tutorial**
=====================================

### Introduction
---------------

Gradient Descent is a fundamental algorithm in machine learning used to minimize the loss function of a model. In 
this tutorial, we will delve into the world of Gradient Descent, exploring its definition, intuition, and practical
implementation.

#### What is Gradient Descent?
Gradient Descent is an iterative optimization algorithm used to minimize a loss function. It works by iteratively 
updating the parameters of the model based on the negative gradient of the loss function with respect to those 
parameters.

**Definition**
---------------

Gradient Descent is defined as:

θ = θ - α * ∇J(θ)

where:

*   θ is the model's parameters
*   α is the learning rate (step size of each update)
*   ∇J(θ) is the gradient of the loss function J with respect to θ

**Intuition**
------------

Gradient Descent can be thought of as an optimization problem, where we want to find the minimum of a function. 
Imagine you're hiking down a mountain, and you want to reach the lowest point. Gradient Descent is like taking 
small steps in the direction of the steepest descent, which will eventually lead you to the minimum.

**Real-world Examples**
---------------------

Gradient Descent has numerous real-world applications, such as:

*   **Loss Function Minimization**: In regression problems, we want to minimize the mean squared error between 
predicted and actual values.
*   **Cost Reduction**: In business, we want to minimize the cost of production while maintaining a certain level 
of quality.

### Mathematical Background
---------------------------

To understand Gradient Descent, we need to dive into some mathematical concepts.

#### Gradient
----------------

The gradient of a function is the rate of change of the function with respect to each input variable. 
Mathematically, it's defined as:

∇f(x) = (∂f/∂x1(x), ..., ∂f/∂xn(x))

where:

*   ∇f(x) is the gradient of the function f at point x
*   ∂f/∂xi(x) is the partial derivative of f with respect to xi at point x

**Intuition**
------------

Think of the gradient as the slope of the function at a given point. It tells us the direction of the steepest 
ascent or descent.

**Mathematical Notation**
----------------------

When writing the gradient, we use the following notation:

∇f(x) = (∂f/